
# 02 — Model Experiments

Train **Prophet** and **XGBoost** independently on the processed dataset, compare their test-set accuracy, and generate a 42-day forward forecast using the winning model.

Pipeline steps executed here:
1. Load raw data → validate → preprocess → engineer features  
2. Train Prophet (time-series split)  
3. Train XGBoost (time-series split)  
4. Compare metrics (MAE / RMSE / MAPE)  
5. Select best model → generate forecast  
6. Visualise forecast with confidence interval


In [ ]:

# ── Setup ─────────────────────────────────────────────────────────────────

import os, sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.data_validation import validate_dataset, detect_optional_columns
from src.preprocessing import clean_dataframe
from src.feature_engineering import create_all_features
from src import forecasting, storage

print("All modules imported successfully.")



## 1. Load, Validate & Preprocess

Load the raw Rossmann data, rename columns to the standard schema, and run the full validation + preprocessing + feature engineering pipeline.


In [ ]:

# ── Load raw data ─────────────────────────────────────────────────────────

train = pd.read_csv("../data/raw/train.csv", parse_dates=["Date"])
store = pd.read_csv("../data/raw/store.csv")

df_raw = train.merge(store, on="Store", how="left").rename(columns={
    "Date": "date",
    "Sales": "sales_qty",
    "Store": "store_id",
    "Promo": "is_promotion",
})

# Keep the standard schema columns only
keep = [c for c in ["date","sales_qty","store_id","is_promotion"] if c in df_raw.columns]
df_raw = df_raw[keep]

# ── Validate ──────────────────────────────────────────────────────────────
df_validated = validate_dataset(df_raw)
print(f"Validated: {len(df_validated):,} rows")

# ── Preprocess ────────────────────────────────────────────────────────────
df_clean = clean_dataframe(df_validated)
print(f"After clean: {len(df_clean):,} rows")

# ── Feature engineering ───────────────────────────────────────────────────
df_features = create_all_features(df_clean)
print(f"Features shape: {df_features.shape}")
print(f"Columns: {list(df_features.columns)}")
df_features.head()



## 2. Train-Test Split Preview

Both models use a **time-based split**: the last 42 days form the held-out test set. No random shuffling is ever performed.


In [ ]:

# ── Visualise the time-based split ───────────────────────────────────────

SPLIT_DAYS = 42
cutoff = df_features["date"].max() - pd.Timedelta(days=SPLIT_DAYS)

agg = df_features.groupby("date")["sales_qty"].sum().reset_index()
train_part = agg[agg["date"] <= cutoff]
test_part  = agg[agg["date"] >  cutoff]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=train_part["date"], y=train_part["sales_qty"],
    mode="lines", name="Train", line={"color": "royalblue"},
))
fig.add_trace(go.Scatter(
    x=test_part["date"], y=test_part["sales_qty"],
    mode="lines", name=f"Test (last {SPLIT_DAYS} days)",
    line={"color": "orange", "dash": "dot"},
))
fig.add_vline(x=str(cutoff.date()), line_dash="dash", line_color="red",
              annotation_text="Split", annotation_position="top left")
fig.update_layout(
    title="Train / Test Split (Time-Based)",
    xaxis_title="Date", yaxis_title="Total Sales",
    template="plotly_white", hovermode="x unified",
)
fig.show()

print(f"Train rows: {len(train_part):,}  |  Test rows: {len(test_part):,}")
print(f"Cutoff date: {cutoff.date()}")



## 3. Train Prophet

Prophet is trained on the **highest-volume store series** as the representative series for seasonality learning.


In [ ]:

# ── Train Prophet ─────────────────────────────────────────────────────────

print("Training Prophet…  (this may take ~30 s)")
prophet_model, prophet_metrics = forecasting.train_prophet(df_features)
_ = storage.save_model(prophet_model, "Prophet")

print("\nProphet test-set metrics:")
for k, v in prophet_metrics.items():
    print(f"  {k:<6}: {v:.4f}")



## 4. Train XGBoost

XGBoost is trained on the **full multi-store dataset** using all engineered lag, rolling, and calendar features. It learns global patterns across all stores simultaneously.


In [ ]:

# ── Train XGBoost ─────────────────────────────────────────────────────────

print("Training XGBoost…")
xgb_model, xgb_metrics, feature_cols = forecasting.train_xgboost(df_features)
_ = storage.save_model(xgb_model, "XGBoost")

print("\nXGBoost test-set metrics:")
for k, v in xgb_metrics.items():
    print(f"  {k:<6}: {v:.4f}")

print(f"\nFeatures used ({len(feature_cols)}):")
print(feature_cols)



## 5. Metrics Comparison & Best Model Selection


In [ ]:

# ── Metrics comparison table ──────────────────────────────────────────────

results = {"Prophet": prophet_metrics, "XGBoost": xgb_metrics}
best_name = forecasting.select_best_model(results)

compare_df = pd.DataFrame([
    {"Model": "Prophet", **prophet_metrics, "Winner": "✅" if best_name == "Prophet" else ""},
    {"Model": "XGBoost", **xgb_metrics,     "Winner": "✅" if best_name == "XGBoost" else ""},
])

def highlight_winner(row):
    if row["Winner"] == "✅":
        return ["background-color:#d4edda; color:#155724"] * len(row)
    return [""] * len(row)

display(compare_df.style
        .apply(highlight_winner, axis=1)
        .format({"MAE": "{:.4f}", "RMSE": "{:.4f}", "MAPE": "{:.2f}"}))

print(f"\n🏆  Best model (lowest MAPE): {best_name}  — MAPE = {results[best_name]['MAPE']:.2f}%")


In [ ]:

# ── Grouped bar: MAE / RMSE / MAPE side-by-side ───────────────────────────

metrics_list = ["MAE", "RMSE", "MAPE"]
fig = go.Figure()
for metric in metrics_list:
    fig.add_trace(go.Bar(
        name=metric,
        x=["Prophet", "XGBoost"],
        y=[results["Prophet"][metric], results["XGBoost"][metric]],
    ))

fig.update_layout(
    barmode="group",
    title="Prophet vs XGBoost — Test-Set Metrics",
    yaxis_title="Score",
    template="plotly_white",
    legend_title="Metric",
)
fig.show()



## 6. XGBoost Feature Importance


In [ ]:

# ── XGBoost Feature Importance ────────────────────────────────────────────

importances = pd.Series(
    xgb_model.feature_importances_,
    index=feature_cols,
).sort_values(ascending=True)

fig = px.bar(
    x=importances.values,
    y=importances.index,
    orientation="h",
    title="XGBoost Feature Importance (gain)",
    labels={"x": "Importance", "y": "Feature"},
    color=importances.values,
    color_continuous_scale="Teal",
    template="plotly_white",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig.show()

print("Top 5 most important features:")
print(importances.tail(5).sort_values(ascending=False))



## 7. Generate 42-Day Forecast (Best Model)


In [ ]:

# ── Generate forecast with the winning model ──────────────────────────────

best_model      = prophet_model if best_name == "Prophet" else xgb_model
best_feat_cols  = [] if best_name == "Prophet" else feature_cols

forecast_df = forecasting.generate_forecast(
    best_model, df_features, best_name, best_feat_cols, horizon=42
)
filepath = storage.save_forecast_output(forecast_df, "model_experiment_forecast.csv")

print(f"Forecast saved to: {filepath}")
print(f"\nForecast rows: {len(forecast_df)}")
forecast_df.head(10)


In [ ]:

## Experiment Summary

| Step | Outcome |
|---|---|
| Data pipeline | Validated → Cleaned → Features engineered |
| Prophet training | Trained on highest-volume series with `is_promotion` regressor |
| XGBoost training | Trained globally across all stores using lag + rolling + calendar features |
| Model selection | Winner determined by lowest test-set MAPE |
| Forecast | 42 days generated with confidence interval |

→ Proceed to **03_accuracy_validation.ipynb** for detailed error analysis.
